# Imports

In [112]:
import numpy as np
import pandas as pd
import joblib
from sklearn.linear_model import LinearRegression
from sklearn_som.som import SOM
from sklearn.model_selection import train_test_split 
import ast
import glob
import re

# Load Data

In [2]:
eigen = pd.read_csv("plink/ref_pca.eigenvec", sep="\t")

In [3]:
eigen.head(10)

,#FID,IID,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,PC11,PC12,PC13,PC14,PC15
0,Luxembourg_Mesolithic.AG,I0001.AG,-0.001732,-0.002810,-0.000781,0.005578,-0.019126,0.009058,0.004034,0.019023,-0.000735,0.010494,-0.006964,0.013426,0.014616,0.007953,0.003552
1,Greece_Koufonisi_Cycladic_EBA.SG,Kou01.SG,-0.003198,-0.008945,0.002482,0.004265,0.019008,-0.000324,0.001644,-0.002295,0.000643,0.009414,0.013041,0.004982,-0.004418,-0.008349,-0.007108
2,Greece_Koufonisi_Cycladic_EBA.SG,Kou03.SG,-0.003041,-0.009028,0.002541,0.001935,0.018752,-0.002034,0.000391,-0.002558,0.001796,0.006719,0.009265,0.007094,0.000050,-0.008312,-0.007226
3,Greece_Logkas_MBA.SG,Log02.SG,-0.003766,-0.009201,0.000576,0.001927,0.007151,-0.000638,0.000163,-0.004511,0.002180,0.004257,0.012402,0.001238,-0.004983,-0.010411,-0.004451
4,Greece_Logkas_MBA.SG,Log04.SG,-0.004005,-0.008548,-0.000638,0.000453,0.003054,-0.001952,0.001757,-0.006693,0.001721,0.000343,0.012880,-0.000423,-0.008249,-0.005648,-0.004317
5,Greece_Manika_EBA.SG,Mik15.SG,-0.003138,-0.009668,0.002858,0.006307,0.019405,-0.000409,0.001012,-0.000972,0.002903,0.011389,0.011573,0.002724,-0.007848,-0.011292,-0.007821
6,Greece_Crete_Kephala_Petras.SG,Pta08.SG,-0.003426,-0.010003,0.003771,0.005044,0.022165,-0.002029,0.000032,-0.002083,0.001877,0.011169,0.010548,0.001443,-0.004830,-0.010912,-0.007495
7,Turkey_Southeast_Cayonu_PPN.SG,cay004.SG,-0.000221,-0.000713,0.000225,0.000052,0.001549,-0.000300,-0.000028,-0.000455,0.000154,0.000879,0.000778,0.001079,0.000460,-0.000306,-0.000169
8,Turkey_Southeast_Cayonu_PPN.SG,cay007.SG,-0.000979,-0.003577,0.001491,0.000557,0.008306,-0.001308,-0.000195,-0.001456,-0.000237,0.002171,0.002963,0.006298,0.003607,-0.003331,-0.003093
9,Turkey_Southeast_Cayonu_PPN.SG,cay008.SG,-0.000166,-0.000611,0.000188,-0.000401,0.001570,-0.000356,0.000110,-0.000648,0.000037,0.000282,0.000813,0.001458,0.001365,-0.000206,-0.000537


# Clean

In [4]:
print(len(eigen))
print(len(eigen["#FID"].unique()))

17573
4281


In [5]:
#eigen = eigen.drop_duplicates(subset=["#FID"])

#eigen.head(10)

#print(len(eigen))

In [6]:
eigen.describe()

,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,PC11,PC12,PC13,PC14,PC15
count,1.757300e+04,1.757300e+04,1.757300e+04,1.757300e+04,1.757300e+04,1.757300e+04,1.757300e+04,1.757300e+04,1.757300e+04,1.757300e+04,1.757300e+04,1.757300e+04,1.757300e+04,1.757300e+04,1.757300e+04
mean,4.110443e-11,-6.885446e-11,-3.397776e-11,-8.271217e-11,1.881568e-11,-1.759344e-11,-1.532683e-11,8.724957e-12,1.594077e-11,-2.219000e-12,-1.364816e-10,1.516163e-10,7.210642e-11,-3.550712e-11,9.206583e-11
std,7.543787e-03,7.543787e-03,7.543787e-03,7.543787e-03,7.543787e-03,7.543787e-03,7.543787e-03,7.543787e-03,7.543788e-03,7.543787e-03,7.543787e-03,7.543787e-03,7.543787e-03,7.543786e-03,7.543784e-03
min,-5.265740e-03,-1.164150e-02,-4.322940e-02,-4.059590e-02,-5.738630e-02,-1.693540e-02,-3.055490e-02,-3.567400e-02,-1.641510e-01,-2.686220e-02,-2.748350e-02,-3.872240e-02,-3.293490e-02,-1.008020e-01,-1.545070e-01
25%,-2.747950e-03,-5.432820e-03,-5.729700e-04,-2.532450e-04,-2.316730e-03,-1.886610e-03,-1.356990e-03,-2.656980e-03,-5.097950e-04,-2.838860e-03,-4.697670e-03,-2.976030e-03,-3.493860e-03,-1.205960e-03,-8.348720e-04
50%,-1.699900e-03,-9.716270e-04,1.370670e-04,3.676390e-04,-4.249960e-06,-4.110150e-05,-3.513500e-05,-2.300850e-04,-1.002660e-05,2.361510e-06,-4.215070e-04,-2.786330e-04,-5.064970e-04,2.692840e-04,1.460480e-04
75%,-3.394320e-04,7.254130e-04,1.897960e-03,2.212360e-03,3.307980e-03,8.972480e-04,1.255000e-03,1.575610e-03,6.448030e-04,2.699220e-03,3.618770e-03,1.940700e-03,1.198780e-03,2.935520e-03,1.746010e-03
max,3.554890e-02,2.190320e-02,1.574850e-02,1.675380e-02,2.722120e-02,9.603950e-02,8.403140e-02,6.043800e-02,1.968440e-02,3.857580e-02,2.150610e-02,4.002290e-02,6.392780e-02,9.683630e-02,1.259240e-01


In [7]:
counts = eigen['#FID'].value_counts()

eigen_filtered = eigen[eigen['#FID'].isin(counts[counts > 1].index)]

In [8]:
print(len(eigen_filtered))

15112


In [10]:
eigen_filtered = eigen_filtered.drop(["IID"],axis=1)

In [11]:
eigen_filtered

,#FID,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,PC11,PC12,PC13,PC14,PC15
1,Greece_Koufonisi_Cycladic_EBA.SG,-0.003198,-0.008945,0.002482,0.004265,0.019008,-0.000324,0.001644,-0.002295,0.000643,0.009414,0.013041,0.004982,-0.004418,-0.008349,-0.007108
2,Greece_Koufonisi_Cycladic_EBA.SG,-0.003041,-0.009028,0.002541,0.001935,0.018752,-0.002034,0.000391,-0.002558,0.001796,0.006719,0.009265,0.007094,0.000050,-0.008312,-0.007226
3,Greece_Logkas_MBA.SG,-0.003766,-0.009201,0.000576,0.001927,0.007151,-0.000638,0.000163,-0.004511,0.002180,0.004257,0.012402,0.001238,-0.004983,-0.010411,-0.004451
4,Greece_Logkas_MBA.SG,-0.004005,-0.008548,-0.000638,0.000453,0.003054,-0.001952,0.001757,-0.006693,0.001721,0.000343,0.012880,-0.000423,-0.008249,-0.005648,-0.004317
7,Turkey_Southeast_Cayonu_PPN.SG,-0.000221,-0.000713,0.000225,0.000052,0.001549,-0.000300,-0.000028,-0.000455,0.000154,0.000879,0.000778,0.001079,0.000460,-0.000306,-0.000169
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17568,Russia_EastSiberia_UstKyakhta_UP.AG,-0.000049,0.000295,-0.000037,0.000044,-0.000092,-0.000055,-0.000254,-0.000179,0.000079,0.000337,-0.000082,-0.000057,-0.000180,-0.000599,-0.000061
17569,Russia_EastSiberia_UstKyakhta_UP.SG,-0.000015,0.000052,-0.000034,0.000035,-0.000026,-0.000090,-0.000028,-0.000056,0.000032,0.000050,-0.000008,-0.000060,-0.000004,-0.000234,-0.000067
17570,Russia_Siberia_Lena_EBA.AG.SG,-0.000087,0.000670,-0.000117,0.000218,-0.000378,-0.000443,-0.000742,-0.000993,-0.000256,0.002175,-0.000290,0.000257,-0.000775,-0.001417,-0.000675
17571,Russia_Siberia_Lena_EBA.AG,-0.000078,0.000597,-0.000132,0.000201,-0.000364,-0.000372,-0.000630,-0.000999,-0.000210,0.001985,-0.000335,0.000377,-0.000662,-0.001417,-0.000686


### Bin the data by time period and region

In [57]:
egfc = eigen_filtered.copy()

In [ ]:
fid = egfc["#FID"].str.replace(r"\.[A-Za-z]+$", "", regex=True)
last_tokens = fid.str.split("_").str[-1]
#print(sorted(last_tokens.unique()))


In [127]:

with open("untitled.txt") as f:
    periods_list = ast.literal_eval(f.read())

periods_list = [re.sub(r"\..*$", "", p) for p in periods_list]
periods_list = sorted({p for p in periods_list if not p.isdigit()}, key=len, reverse=True)
periods_list = [re.escape(p) for p in periods_list]
periods = r"(?:{})".format("|".join(periods_list))

periods_list = sorted(periods_list, key=len, reverse=True)
periods = r"(?:{})".format("|".join(map(re.escape, periods_list)))

#periods

In [119]:
fid = egfc["#FID"].str.replace(r"\.[A-Za-z]+$", "", regex=True)
egfc["Period"] = fid.str.extract(fr"_({periods})$")
egfc["Name"] = fid.str.replace(fr"_({periods})$", "", regex=True)

In [123]:
egfc[['#FID', 'Period', 'Name']].head(30)

,#FID,Period,Name
1,Greece_Koufonisi_Cycladic_EBA.SG,EBA,Greece_Koufonisi_Cycladic
2,Greece_Koufonisi_Cycladic_EBA.SG,EBA,Greece_Koufonisi_Cycladic
3,Greece_Logkas_MBA.SG,MBA,Greece_Logkas
4,Greece_Logkas_MBA.SG,MBA,Greece_Logkas
7,Turkey_Southeast_Cayonu_PPN.SG,PPN,Turkey_Southeast_Cayonu
8,Turkey_Southeast_Cayonu_PPN.SG,PPN,Turkey_Southeast_Cayonu
9,Turkey_Southeast_Cayonu_PPN.SG,PPN,Turkey_Southeast_Cayonu
10,Turkey_Southeast_Cayonu_PPN.SG,PPN,Turkey_Southeast_Cayonu
11,Turkey_Southeast_Cayonu_PPN.SG,PPN,Turkey_Southeast_Cayonu
12,Turkey_Southeast_Cayonu_PPN.SG,PPN,Turkey_Southeast_Cayonu


In [124]:
egfc[egfc['Period'] == 'Modern']

,#FID,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,PC11,PC12,PC13,PC14,PC15,Period,Name,Clean_FID
46,Hungary_Langobard.AG,-0.002328,-0.004975,-0.000300,0.001215,-0.004107,0.000793,-0.001936,-0.003252,-0.000904,-0.011221,-0.010540,-0.000143,-0.002776,0.007499,0.006122,Modern,Hungary_Langobard,Hungary_Langobard
48,Hungary_Langobard.AG,-0.001306,-0.003400,0.000165,0.000705,-0.002314,0.000432,-0.001541,-0.002636,0.000022,-0.007683,-0.006195,-0.001381,-0.000652,0.004621,0.004125,Modern,Hungary_Langobard,Hungary_Langobard
49,Hungary_Langobard.AG,-0.002724,-0.005281,-0.000719,0.000677,-0.006762,0.000449,-0.001035,-0.004361,0.000413,-0.009639,-0.005157,-0.002160,-0.004693,0.008203,0.005519,Modern,Hungary_Langobard,Hungary_Langobard
50,Hungary_Langobard_o2.AG,-0.002513,-0.006031,0.000347,0.001157,-0.000286,-0.000233,-0.001693,-0.005131,-0.000713,-0.008034,-0.007101,0.000143,-0.001374,0.004284,0.002062,Modern,Hungary_Langobard_o2,Hungary_Langobard_o2
51,Hungary_Langobard_o1.AG,-0.002659,-0.007360,0.001653,0.003383,0.009930,-0.000838,-0.002776,-0.003263,0.000075,-0.007033,-0.006823,-0.001902,0.002032,0.001524,0.000616,Modern,Hungary_Langobard_o1,Hungary_Langobard_o1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17498,Italy_Tarquinia_Etruscan.SG,-0.000112,-0.000224,-0.000010,0.000029,-0.000018,0.000028,-0.000023,0.000081,-0.000014,-0.000094,0.000130,-0.000226,0.000177,0.000048,-0.000160,Modern,Italy_Tarquinia_Etruscan,Italy_Tarquinia_Etruscan
17499,Japan_Hamanaka_Okhotsk.SG,-0.002918,0.019377,0.005663,0.005508,-0.002420,-0.006605,-0.021295,-0.003137,-0.000365,0.017556,-0.005530,-0.018258,0.010779,0.006891,0.007488,Modern,Japan_Hamanaka_Okhotsk,Japan_Hamanaka_Okhotsk
17500,Japan_Hamanaka_Okhotsk.SG,-0.000949,0.006886,0.001691,0.001821,-0.001545,-0.002159,-0.006184,-0.002962,0.000625,0.010770,-0.000140,-0.002168,0.001100,0.001062,0.001986,Modern,Japan_Hamanaka_Okhotsk,Japan_Hamanaka_Okhotsk
17501,Germany_Beethoven.SG,-0.000105,-0.000266,-0.000034,0.000033,-0.000190,0.000122,0.000131,-0.000089,-0.000026,-0.000261,0.000353,-0.000099,0.000016,0.000294,0.000189,Modern,Germany_Beethoven,Germany_Beethoven


In [128]:
egfc.to_csv("out.csv")

In [58]:
odf = pd.read_csv("out_with_regions_and_periods_updated.csv", sep=",")

In [59]:
odf.drop(columns=['Unnamed: 0'], inplace=True)

In [76]:
odf.drop(['Period'], axis=1)

,#FID,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,PC11,PC12,PC13,PC14,PC15,Name,Clean_FID,Broad_Period,Region
0,Greece_Koufonisi_Cycladic_EBA.SG,-0.003198,-0.008945,0.002482,0.004265,0.019008,-0.000324,0.001644,-0.002295,0.000643,0.009414,0.013041,0.004982,-0.004418,-0.008349,-0.007108,Greece_Koufonisi_Cycladic,Greece_Koufonisi_Cycladic_EBA,Bronze-Age,Southern European
1,Greece_Koufonisi_Cycladic_EBA.SG,-0.003041,-0.009028,0.002541,0.001935,0.018752,-0.002034,0.000391,-0.002558,0.001796,0.006719,0.009265,0.007094,0.000050,-0.008312,-0.007226,Greece_Koufonisi_Cycladic,Greece_Koufonisi_Cycladic_EBA,Bronze-Age,Southern European
2,Greece_Logkas_MBA.SG,-0.003766,-0.009201,0.000576,0.001927,0.007151,-0.000638,0.000163,-0.004511,0.002180,0.004257,0.012402,0.001238,-0.004983,-0.010411,-0.004451,Greece_Logkas,Greece_Logkas_MBA,Bronze-Age,Southern European
3,Greece_Logkas_MBA.SG,-0.004005,-0.008548,-0.000638,0.000453,0.003054,-0.001952,0.001757,-0.006693,0.001721,0.000343,0.012880,-0.000423,-0.008249,-0.005648,-0.004317,Greece_Logkas,Greece_Logkas_MBA,Bronze-Age,Southern European
4,Turkey_Southeast_Cayonu_PPN.SG,-0.000221,-0.000713,0.000225,0.000052,0.001549,-0.000300,-0.000028,-0.000455,0.000154,0.000879,0.000778,0.001079,0.000460,-0.000306,-0.000169,Turkey_Southeast_Cayonu,Turkey_Southeast_Cayonu_PPN,Neolithic,Middle Eastern
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15107,Russia_EastSiberia_UstKyakhta_UP.AG,-0.000049,0.000295,-0.000037,0.000044,-0.000092,-0.000055,-0.000254,-0.000179,0.000079,0.000337,-0.000082,-0.000057,-0.000180,-0.000599,-0.000061,Russia_EastSiberia_UstKyakhta,Russia_EastSiberia_UstKyakhta_UP,Palaeolithic,Eastern European
15108,Russia_EastSiberia_UstKyakhta_UP.SG,-0.000015,0.000052,-0.000034,0.000035,-0.000026,-0.000090,-0.000028,-0.000056,0.000032,0.000050,-0.000008,-0.000060,-0.000004,-0.000234,-0.000067,Russia_EastSiberia_UstKyakhta,Russia_EastSiberia_UstKyakhta_UP,Palaeolithic,Eastern European
15109,Russia_Siberia_Lena_EBA.AG.SG,-0.000087,0.000670,-0.000117,0.000218,-0.000378,-0.000443,-0.000742,-0.000993,-0.000256,0.002175,-0.000290,0.000257,-0.000775,-0.001417,-0.000675,Russia_Siberia_Lena.G,Russia_Siberia_Lena_EBA,Bronze-Age,Eastern European
15110,Russia_Siberia_Lena_EBA.AG,-0.000078,0.000597,-0.000132,0.000201,-0.000364,-0.000372,-0.000630,-0.000999,-0.000210,0.001985,-0.000335,0.000377,-0.000662,-0.001417,-0.000686,Russia_Siberia_Lena,Russia_Siberia_Lena_EBA,Bronze-Age,Eastern European


In [77]:
odf.head(5)

,#FID,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,...,PC11,PC12,PC13,PC14,PC15,Period,Name,Clean_FID,Broad_Period,Region
0,Greece_Koufonisi_Cycladic_EBA.SG,-0.003198,-0.008945,0.002482,0.004265,0.019008,-0.000324,0.001644,-0.002295,0.000643,...,0.013041,0.004982,-0.004418,-0.008349,-0.007108,EBA,Greece_Koufonisi_Cycladic,Greece_Koufonisi_Cycladic_EBA,Bronze-Age,Southern European
1,Greece_Koufonisi_Cycladic_EBA.SG,-0.003041,-0.009028,0.002541,0.001935,0.018752,-0.002034,0.000391,-0.002558,0.001796,...,0.009265,0.007094,0.000050,-0.008312,-0.007226,EBA,Greece_Koufonisi_Cycladic,Greece_Koufonisi_Cycladic_EBA,Bronze-Age,Southern European
2,Greece_Logkas_MBA.SG,-0.003766,-0.009201,0.000576,0.001927,0.007151,-0.000638,0.000163,-0.004511,0.002180,...,0.012402,0.001238,-0.004983,-0.010411,-0.004451,MBA,Greece_Logkas,Greece_Logkas_MBA,Bronze-Age,Southern European
3,Greece_Logkas_MBA.SG,-0.004005,-0.008548,-0.000638,0.000453,0.003054,-0.001952,0.001757,-0.006693,0.001721,...,0.012880,-0.000423,-0.008249,-0.005648,-0.004317,MBA,Greece_Logkas,Greece_Logkas_MBA,Bronze-Age,Southern European
4,Turkey_Southeast_Cayonu_PPN.SG,-0.000221,-0.000713,0.000225,0.000052,0.001549,-0.000300,-0.000028,-0.000455,0.000154,...,0.000778,0.001079,0.000460,-0.000306,-0.000169,PPN,Turkey_Southeast_Cayonu,Turkey_Southeast_Cayonu_PPN,Neolithic,Middle Eastern


In [78]:
odf[odf['Region'] == 'Other']

,#FID,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,...,PC11,PC12,PC13,PC14,PC15,Period,Name,Clean_FID,Broad_Period,Region
8730,BIR.SG,-0.001116,1.072690e-02,0.009640,-0.023796,0.003542,-0.000822,0.006190,0.029289,-0.000201,...,-0.011734,-0.013205,-0.022584,-0.016365,-0.003330,BIR,SG,BIR,Historical/Modern,Other
8731,BIR.SG,-0.001246,1.088800e-02,0.010430,-0.023779,0.003020,-0.001550,0.005301,0.030241,-0.000820,...,-0.011700,-0.014992,-0.025151,-0.020272,-0.005165,BIR,SG,BIR,Historical/Modern,Other
8732,BIR.SG,-0.001102,1.084840e-02,0.010140,-0.024295,0.003012,-0.000786,0.005264,0.031041,0.000255,...,-0.012499,-0.015896,-0.026299,-0.019167,-0.002862,BIR,SG,BIR,Historical/Modern,Other
8733,BIR.SG,-0.000989,1.061530e-02,0.009630,-0.024175,0.003884,-0.000607,0.004294,0.030926,0.000127,...,-0.014575,-0.016712,-0.025772,-0.020494,-0.003215,BIR,SG,BIR,Historical/Modern,Other
8734,BIR.SG,-0.001155,1.058000e-02,0.010429,-0.023653,0.003687,-0.000713,0.005988,0.028130,0.000052,...,-0.013189,-0.011642,-0.021010,-0.014628,-0.004725,BIR,SG,BIR,Historical/Modern,Other
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13199,StHelena_RupertsValley_African.SG,0.004749,2.608250e-05,-0.000171,0.000299,-0.000441,-0.000157,0.000153,0.000118,-0.000462,...,0.000096,0.000095,0.000118,-0.001850,0.001754,A,StHelena_RupertsValleyfrican,StHelena_RupertsValley_African,Other,Other
13200,StHelena_RupertsValley_African.SG,0.004810,-2.273400e-05,-0.000154,0.000217,-0.000395,-0.000134,0.000258,0.000276,-0.000290,...,0.000440,0.000281,-0.000327,-0.001314,0.002038,A,StHelena_RupertsValleyfrican,StHelena_RupertsValley_African,Other,Other
13201,StHelena_RupertsValley_African.SG,0.004555,-5.012110e-07,0.000093,0.000209,-0.000198,-0.000110,-0.000340,-0.000066,0.000669,...,0.000702,0.000157,0.000853,-0.001570,0.000743,A,StHelena_RupertsValleyfrican,StHelena_RupertsValley_African,Other,Other
13202,StHelena_RupertsValley_African.SG,0.002586,-3.183790e-05,-0.000018,0.000166,-0.000255,0.000305,0.000340,-0.000049,0.000357,...,0.000437,-0.000012,0.000682,-0.000490,0.000965,A,StHelena_RupertsValleyfrican,StHelena_RupertsValley_African,Other,Other


In [79]:
odf[odf['Broad_Period'] == 'Other']

,#FID,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,...,PC11,PC12,PC13,PC14,PC15,Period,Name,Clean_FID,Broad_Period,Region
79,Russia_Afanasievo.AG,-0.001114,-0.001649,-0.001175,-0.003628,-0.006149,-0.002763,-0.000934,-0.006121,-0.000384,...,-0.001572,0.000549,-0.000462,-0.000170,-0.001236,A,Russiafanasievo,Russia_Afanasievo,Other,Eastern European
105,Poland_GlobularAmphora.AG,-0.001125,-0.003134,0.001008,0.004541,0.001106,0.003182,-0.001221,0.004030,-0.000590,...,-0.006588,-0.004136,-0.000043,0.000340,0.001012,A,Poland_Globularmphora,Poland_GlobularAmphora,Other,Eastern European
153,USA_CA_LHolocene_Barbareno_Chumash.AG,-0.002994,0.013914,-0.030134,0.000447,0.005623,0.001928,0.005830,-0.003665,-0.000992,...,-0.005933,0.020139,-0.018995,-0.004009,-0.001817,A,USolocene_Barbareno_Chumash,USA_CA_LHolocene_Barbareno_Chumash,Other,Native American
154,USA_CA_LHolocene_Barbareno_Chumash.AG,-0.002891,0.013465,-0.028973,-0.000517,0.005498,0.002557,0.005201,-0.002224,-0.000152,...,-0.003384,0.019997,-0.018353,-0.003241,-0.003075,A,USolocene_Barbareno_Chumash,USA_CA_LHolocene_Barbareno_Chumash,Other,Native American
155,USA_CA_LHolocene_Barbareno_Chumash.AG,-0.001237,0.006223,-0.014216,0.000024,0.002380,0.001164,0.003244,-0.002614,-0.000755,...,-0.003518,0.014984,-0.012604,-0.002985,-0.000792,A,USolocene_Barbareno_Chumash,USA_CA_LHolocene_Barbareno_Chumash,Other,Native American
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15007,Mexico_CDMX_EarlyColonial_African.SG.AG,0.002036,-0.000059,0.000039,-0.000167,-0.000059,-0.000010,-0.000091,0.000347,0.000320,...,0.002045,-0.000444,0.002421,-0.000921,-0.001755,Early,Mexico_CDMXfrican.SG,Mexico_CDMX_EarlyColonial_African,Other,Native American
15008,Mexico_CDMX_EarlyColonial_African.SG.AG,0.022793,0.000302,-0.000429,0.000136,-0.000552,-0.000972,-0.000999,0.002427,-0.002514,...,0.014602,0.000920,0.015218,-0.022067,0.008071,Early,Mexico_CDMXfrican.SG,Mexico_CDMX_EarlyColonial_African,Other,Native American
15009,Mexico_CDMX_EarlyColonial_African.SG.AG,0.002678,0.000080,0.000208,-0.000293,0.000005,-0.000012,0.000156,0.000409,-0.000350,...,0.003612,-0.000010,0.003781,-0.002474,-0.000020,Early,Mexico_CDMXfrican.SG,Mexico_CDMX_EarlyColonial_African,Other,Native American
15010,Mexico_CDMX_EarlyColonial_African.SG.AG,0.026263,0.000024,-0.000272,0.000012,-0.000537,-0.000844,0.000110,0.001526,-0.002006,...,0.012996,0.001929,0.015749,-0.020068,0.006590,Early,Mexico_CDMXfrican.SG,Mexico_CDMX_EarlyColonial_African,Other,Native American


In [80]:
odf[odf['Name'].isna()]

,#FID,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,...,PC11,PC12,PC13,PC14,PC15,Period,Name,Clean_FID,Broad_Period,Region


In [81]:
mask = odf['Name'].isna()

odf.loc[mask, 'Name'] = odf.loc[mask, '#FID'].str.split('.').str[-1]

In [82]:
print(odf.iloc[8585])

#FID                       ASW.SG
PC1                      0.022637
PC2                     -0.002196
PC3                     -0.000225
PC4                      0.001821
PC5                     -0.003353
PC6                     -0.000198
PC7                      0.002035
PC8                      0.000696
PC9                      0.005965
PC10                     0.002023
PC11                     0.004511
PC12                     -0.00068
PC13                    -0.007919
PC14                    -0.001233
PC15                     0.003106
Period                        ASW
Name                           SG
Clean_FID                     ASW
Broad_Period    Historical/Modern
Region               West African
Name: 8585, dtype: object


In [83]:
other = odf.loc[odf['Broad_Period'] == 'Other', 'Name'].tolist()

In [84]:
other

['Russiafanasievo',
 'Poland_Globularmphora',
 'USolocene_Barbareno_Chumash',
 'USolocene_Barbareno_Chumash',
 'USolocene_Barbareno_Chumash',
 'Turkeyegean_Bodrum_Halikarnassos',
 'Turkeyegean_Bodrum_Halikarnassos',
 'India_Roopkund',
 'India_Roopkund',
 'India_Roopkund',
 'India_Roopkund',
 'India_Roopkund',
 'India_Roopkund',
 'India_Roopkund',
 'Russiafanasievo',
 'India_Roopkund',
 'India_Roopkund',
 'India_Roopkund',
 'rmenia_KarmirBlur_Urartian',
 'rmenia_KarmirBlur_Urartian',
 'Russiafanasievo',
 'Russiafanasievo',
 'Spain_Roman_ofrica',
 'Turkmenistan_C_Tepenau',
 'Turkmenistan_C_Tepenau',
 'Turkmenistan_C_Tepenau',
 'Turkey_BlackSea_Samsun',
 'Turkey_BlackSea_Samsun',
 'Italy_Sicily_Himera_480BCE',
 'USK_Paleoleut',
 'Russia_UstBelayangara',
 'ustria_Gravettian',
 'Russiafanasievo',
 'Russiafanasievo',
 'HungaryC_Tiszapolgar',
 'HungaryC_Tiszapolgar',
 'HungaryC_Tiszapolgar',
 'Poland_Globularmphora',
 'Poland_Globularmphora_brother.I2407_son.I2433_son.I2440',
 'Poland_Globula

In [85]:
odf.to_csv("Final_out.csv")

In [86]:
fodf = pd.read_csv("Final_out.csv", sep=",")

In [87]:
fodf.drop(columns=['Unnamed: 0'], inplace=True)

In [91]:
fodf=fodf.drop(['Period'],axis=1)

In [92]:
fodf.head(5)

,#FID,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,PC11,PC12,PC13,PC14,PC15,Name,Clean_FID,Broad_Period,Region
0,Greece_Koufonisi_Cycladic_EBA.SG,-0.003198,-0.008945,0.002482,0.004265,0.019008,-0.000324,0.001644,-0.002295,0.000643,0.009414,0.013041,0.004982,-0.004418,-0.008349,-0.007108,Greece_Koufonisi_Cycladic,Greece_Koufonisi_Cycladic_EBA,Bronze-Age,Southern European
1,Greece_Koufonisi_Cycladic_EBA.SG,-0.003041,-0.009028,0.002541,0.001935,0.018752,-0.002034,0.000391,-0.002558,0.001796,0.006719,0.009265,0.007094,0.000050,-0.008312,-0.007226,Greece_Koufonisi_Cycladic,Greece_Koufonisi_Cycladic_EBA,Bronze-Age,Southern European
2,Greece_Logkas_MBA.SG,-0.003766,-0.009201,0.000576,0.001927,0.007151,-0.000638,0.000163,-0.004511,0.002180,0.004257,0.012402,0.001238,-0.004983,-0.010411,-0.004451,Greece_Logkas,Greece_Logkas_MBA,Bronze-Age,Southern European
3,Greece_Logkas_MBA.SG,-0.004005,-0.008548,-0.000638,0.000453,0.003054,-0.001952,0.001757,-0.006693,0.001721,0.000343,0.012880,-0.000423,-0.008249,-0.005648,-0.004317,Greece_Logkas,Greece_Logkas_MBA,Bronze-Age,Southern European
4,Turkey_Southeast_Cayonu_PPN.SG,-0.000221,-0.000713,0.000225,0.000052,0.001549,-0.000300,-0.000028,-0.000455,0.000154,0.000879,0.000778,0.001079,0.000460,-0.000306,-0.000169,Turkey_Southeast_Cayonu,Turkey_Southeast_Cayonu_PPN,Neolithic,Middle Eastern


In [95]:
fodf.groupby(['Region', 'Broad_Period']).size().reset_index(name='count')

,Region,Broad_Period,count
0,Carribean,Historical/Modern,94
1,Carribean,Other,3
2,Caucasian,Historical/Modern,2
3,Caucasian,Other,19
4,Central African,Historical/Modern,46
...,...,...,...
106,Western European,Historical/Modern,280
107,Western European,Iron-Age,128
108,Western European,Mesolithic,61
109,Western European,Neolithic,354


In [115]:
import pandas as pd
import numpy as np


pc_cols = [c for c in fodf.columns if c.startswith("PC")]

centroids = (
    fodf.groupby(["Region", "Broad_Period"])[pc_cols]
      .mean()
      .reset_index()
)

centroids["Population_Label"] = (
    centroids["Region"] + " | " + centroids["Broad_Period"]
)

M = centroids[pc_cols].to_numpy()

labels = centroids["Population_Label"].to_numpy()

regions = centroids["Region"].to_numpy()
periods = centroids["Broad_Period"].to_numpy()

print("Centroid dataframe:")
centroids.head(5)

print("\nMatrix shape (G x d):")
print(M.shape)

print("\nPopulation labels:")
print(labels)

Centroid dataframe:

Matrix shape (G x d):
(111, 15)

Population labels:
['Carribean | Historical/Modern' 'Carribean | Other'
 'Caucasian | Historical/Modern' 'Caucasian | Other'
 'Central African | Historical/Modern' 'Central African | Other'
 'Central Asian | Bronze-Age' 'Central Asian | Historical/Modern'
 'Central Asian | Iron-Age' 'Central Asian | Neolithic'
 'Central Asian | Other' 'East Asian | Bronze-Age'
 'East Asian | Historical/Modern' 'East Asian | Iron-Age'
 'East Asian | Mesolithic' 'East Asian | Neolithic' 'East Asian | Other'
 'East Asian | Palaeolithic' 'East Asian | Unknown'
 'Eastern African | Historical/Modern' 'Eastern African | Iron-Age'
 'Eastern African | Neolithic' 'Eastern African | Other'
 'Eastern European | Bronze-Age' 'Eastern European | Copper'
 'Eastern European | Historical/Modern' 'Eastern European | Iron-Age'
 'Eastern European | Mesolithic' 'Eastern European | Neolithic'
 'Eastern European | Other' 'Eastern European | Palaeolithic'
 'Horn of African 

In [116]:
centroids.head(5)

,Region,Broad_Period,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,PC11,PC12,PC13,PC14,PC15,Population_Label
0,Carribean,Historical/Modern,0.029959,-0.001008,-0.001039,0.001216,-0.002107,-0.000854,0.000648,-0.000017,0.011320,-0.000218,0.000626,-0.002115,-0.003393,0.003847,0.000983,Carribean | Historical/Modern
1,Carribean,Other,0.006966,-0.000029,-0.000166,0.000081,-0.000194,-0.000124,-0.000092,0.000413,0.000651,0.000062,0.000930,0.000491,0.000705,-0.001711,0.001876,Carribean | Other
2,Caucasian,Historical/Modern,-0.003483,-0.006051,0.000743,-0.011089,0.002028,-0.009108,0.001844,-0.009560,0.001549,-0.003513,0.015338,0.007014,0.011169,0.005528,0.000821,Caucasian | Historical/Modern
3,Caucasian,Other,-0.003593,-0.005797,0.001657,-0.008390,0.005710,-0.008334,0.001114,-0.006859,0.001207,0.000571,0.015389,0.009037,0.011078,0.007646,0.001525,Caucasian | Other
4,Central African,Historical/Modern,0.032576,0.000186,-0.000564,-0.000420,-0.001310,0.002257,-0.002376,0.002032,-0.084232,-0.002830,0.010471,-0.005962,-0.007272,-0.016917,0.063145,Central African | Historical/Modern


In [117]:
M.shape

(111, 15)

In [118]:
centroids.to_csv("centroids.csv", index=False)
np.save("centroid_matrix.npy", M)
np.savetxt("population_labels.txt", labels, fmt="%s")

print("Saved files:")
print("centroids.csv")
print("centroid_matrix.npy")
print("population_labels.txt")

Saved files:
centroids.csv
centroid_matrix.npy
population_labels.txt


K-nearest neighbour, LG regression, SOM were trained all had poor accuracy ~30-40%. Issue is llarge dataset ~17k samples with ~4k classes with low distribution, some classes only have one instances. So the solution was to bin them by period and region.